In [1]:
# ╔═══════════════════════════════════════════════════════════════════════╗
# ║  MHIPEX v10 — hmBERT + mBERT ensemble                               ║
# ║  Fix: Replace XLM-R-base (dies every run) with mBERT (stable)       ║
# ║  Both use WordPiece → no embedding-resize gradient spike             ║
# ║  Expected MR: 0.56–0.62 | Runtime: ~1.5h on T4 x2                  ║
# ╚═══════════════════════════════════════════════════════════════════════╝

import os, math, json, re, random, copy, gc, time
os.environ["PYTORCH_ALLOC_CONF"]     = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import subprocess
subprocess.run(["pip","install","-q","transformers==4.40.2","accelerate","scikit-learn","tqdm"],check=True)
print("✅ Packages ready")

from pathlib import Path
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from transformers import AutoModel, AutoTokenizer, get_cosine_schedule_with_warmup
from sklearn.metrics import recall_score, classification_report
from tqdm import tqdm
import urllib.request

SEED = 42
def set_seed(s=SEED):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
set_seed()
torch.backends.cuda.matmul.allow_tf32 = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_GPU  = torch.cuda.device_count()
print(f"Device : {DEVICE} | GPUs : {N_GPU} | CUDA : {torch.version.cuda}")

AT_MAP     = {"FALSE":0,"PROBABLE":1,"TRUE":2}
ISAT_MAP   = {"FALSE":0,"TRUE":1}
AT_NAMES   = ["FALSE","PROBABLE","TRUE"]
ISAT_NAMES = ["FALSE","TRUE"]
SPECIAL    = ["<P>","</P>","<L>","</L>"]
AT_W       = torch.tensor([0.80,1.50,2.40], dtype=torch.float32)
ISAT_W     = torch.tensor([0.70,2.60],      dtype=torch.float32)

# ── Model configs ─────────────────────────────────────────────────────────────
# KEY CHANGE: XLM-R-base → mBERT
# WHY: XLM-R uses SentencePiece (250K vocab). Adding 4 new tokens causes
#      uninitialized embedding rows → exploding gradients on backward pass 1
#      → GradScaler kills all updates → loss=0.0 → dead model.
# mBERT uses WordPiece (119K vocab), same architecture as hmBERT.
# Adding tokens to WordPiece models is safe → no spike.
MODEL_CFG = {
    "hmbert": {
        "name":   "dbmdz/bert-base-historic-multilingual-cased",
        "bs":32, "accum":2, "lr":8e-6, "decay":0.90,
        "ckpt":False, "pat":8, "ep":35, "wu":0.12, "maxlen":256, "drop":0.15, "clip":0.5,
    },
    "mbert": {
        "name":   "google/bert-base-multilingual-cased",  # ← stable replacement
        "bs":32, "accum":2, "lr":8e-6, "decay":0.90,
        "ckpt":False, "pat":8, "ep":35, "wu":0.12, "maxlen":256, "drop":0.15, "clip":0.5,
    },
}

WD       = 0.01
DATA_DIR = Path("data");  DATA_DIR.mkdir(exist_ok=True)
PROC_DIR = Path("proc");  PROC_DIR.mkdir(exist_ok=True)
OUT_DIR  = Path("out");   OUT_DIR.mkdir(exist_ok=True)

# ── Data ──────────────────────────────────────────────────────────────────────
BASE  = "https://raw.githubusercontent.com/hipe-eval/HIPE-2026-data/main/data/sandbox/"
FILES = ["en-train.jsonl","fr-train.jsonl","de-train.jsonl",
         "en-dev.jsonl","fr-dev.jsonl","de-dev.jsonl"]
print("\n── Data ─────────────────────────────────────────────────────────────")
for f in FILES:
    p = DATA_DIR/f
    if not p.exists():
        urllib.request.urlretrieve(BASE+f,p); print(f"  Downloaded {f}")
    else: print(f"  {f} (cached)")
print("✅ Data ready\n")

def clean(t,n=900): return re.sub(r"\s+"," ",re.sub(r"[\x00-\x1f\x7f-\x9f]"," ",t)).strip()[:n]
def build(text,pers,locs):
    p=clean(pers[0]) if pers else "UNKNOWN"
    l=clean(locs[0]) if locs else "UNKNOWN"
    return f"<P> {p} </P> <L> {l} </L> {clean(text,800)}"
def load_jsonl(path,lang):
    recs=[]
    for line in open(path,encoding="utf-8"):
        doc=json.loads(line)
        for pair in doc.get("sampled_pairs",[]):
            at=pair.get("at","FALSE"); isat=pair.get("isAt","FALSE")
            if at not in AT_MAP or isat not in ISAT_MAP: continue
            recs.append({"text":build(doc["text"],pair["pers_mentions_list"],
                pair["loc_mentions_list"]),"at":AT_MAP[at],"isat":ISAT_MAP[isat],"lang":lang})
    return recs

print("── Dataset ──────────────────────────────────────────────────────────")
for split,fmap in [("train",{"en":"en-train","fr":"fr-train","de":"de-train"}),
                   ("dev",  {"en":"en-dev",  "fr":"fr-dev",  "de":"de-dev"})]:
    all_r=[]
    for lang,fname in fmap.items():
        r=load_jsonl(DATA_DIR/f"{fname}.jsonl",lang); all_r.extend(r)
        print(f"  {lang} {split:5s}: {len(r):,}")
    with open(PROC_DIR/f"{split}.jsonl","w",encoding="utf-8") as f:
        for r in all_r: f.write(json.dumps(r,ensure_ascii=False)+"\n")
    print(f"✅ {split}: {len(all_r):,}\n")

class HIPEDataset(Dataset):
    def __init__(self,path,tok,maxlen=256):
        self.data=[json.loads(l) for l in open(path,encoding="utf-8")]
        self.tok=tok; self.maxlen=maxlen
    def __len__(self): return len(self.data)
    def __getitem__(self,i):
        d=self.data[i]
        enc=self.tok(d["text"],max_length=self.maxlen,truncation=True,
                     padding="max_length",return_tensors="pt")
        return {"input_ids":enc["input_ids"].squeeze(0),
                "attention_mask":enc["attention_mask"].squeeze(0),
                "at_label":torch.tensor(d["at"],dtype=torch.long),
                "isat_label":torch.tensor(d["isat"],dtype=torch.long)}

def balanced_sampler(ds):
    labels=np.array([r["at"] for r in ds.data])
    counts=np.bincount(labels,minlength=3).astype(float)
    return WeightedRandomSampler([1.0/counts[l] for l in labels],len(labels),replacement=True)

class FocalLoss(nn.Module):
    def __init__(self,weight=None,gamma=2.0):
        super().__init__(); self.gamma=gamma
        self.register_buffer("weight",weight)
    def forward(self,logits,targets):
        ce=F.cross_entropy(logits,targets,weight=self.weight,reduction="none")
        return (((1-torch.exp(-ce))**self.gamma)*ce).mean()

class MHIPEXModel(nn.Module):
    def __init__(self,name,n_new_tok,dropout=0.15,grad_ckpt=False):
        super().__init__()
        self.encoder=AutoModel.from_pretrained(name)
        self.encoder.resize_token_embeddings(self.encoder.config.vocab_size+n_new_tok)
        if grad_ckpt: self.encoder.gradient_checkpointing_enable()
        h=self.encoder.config.hidden_size
        self.layer_norm=nn.LayerNorm(h); self.dropout=nn.Dropout(dropout)
        self.head_at=nn.Sequential(nn.Linear(h,h//2),nn.GELU(),nn.Dropout(dropout),nn.Linear(h//2,3))
        self.head_isat=nn.Sequential(nn.Linear(h,h//2),nn.GELU(),nn.Dropout(dropout),nn.Linear(h//2,2))
    def forward(self,input_ids,attention_mask):
        out=self.encoder(input_ids=input_ids,attention_mask=attention_mask)
        cls=self.layer_norm(self.dropout(out.last_hidden_state[:,0,:]))
        return self.head_at(cls),self.head_isat(cls)

def layerwise_groups(model,base_lr,decay):
    raw=model.module if hasattr(model,"module") else model; enc=raw.encoder; groups=[]
    layers=getattr(getattr(enc,"encoder",enc),"layer",None)
    if layers:
        n=len(layers)
        for i,layer in enumerate(layers):
            groups.append({"params":list(layer.parameters()),"lr":base_lr*(decay**(n-1-i))})
    if hasattr(enc,"embeddings"):
        groups.append({"params":list(enc.embeddings.parameters()),
                       "lr":base_lr*(decay**(len(layers) if layers else 12))})
    for head in [raw.head_at,raw.head_isat,raw.layer_norm]:
        groups.append({"params":list(head.parameters()),"lr":base_lr})
    return groups

def macro_recall(at_t,at_p,is_t,is_p):
    r1=recall_score(at_t,at_p,average="macro",zero_division=0)
    r2=recall_score(is_t,is_p,average="macro",zero_division=0)
    return round((r1+r2)/2,4),round(r1,4),round(r2,4)

def train_model(model_key):
    cfg=MODEL_CFG[model_key]; eff_bs=cfg["bs"]*cfg["accum"]
    print(f"\n{'='*64}")
    print(f"  {model_key.upper()} | {cfg['name']}")
    print(f"  LR={cfg['lr']} | EffBS={eff_bs} | Pat={cfg['pat']} | clip={cfg['clip']}")
    print(f"{'='*64}\n")
    gc.collect(); torch.cuda.empty_cache(); torch.cuda.synchronize()
    print(f"  GPU free: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB"); set_seed(); t0=time.time()

    tok=AutoTokenizer.from_pretrained(cfg["name"])
    n=tok.add_special_tokens({"additional_special_tokens":SPECIAL})
    for st in SPECIAL:
        ids=tok.encode(st,add_special_tokens=False)
        print(f"    '{st}'→{ids} ({'✅' if len(ids)==1 else '⚠ split'})")

    tr_ds=HIPEDataset(PROC_DIR/"train.jsonl",tok,cfg["maxlen"])
    dv_ds=HIPEDataset(PROC_DIR/"dev.jsonl",  tok,cfg["maxlen"])
    TL=DataLoader(tr_ds,batch_size=cfg["bs"],sampler=balanced_sampler(tr_ds),num_workers=2,pin_memory=True)
    DL=DataLoader(dv_ds,batch_size=cfg["bs"],shuffle=False,num_workers=2,pin_memory=True)

    model=MHIPEXModel(cfg["name"],n_new_tok=n,dropout=cfg["drop"],grad_ckpt=cfg["ckpt"]).to(DEVICE)
    if N_GPU>1: model=nn.DataParallel(model); print(f"  DataParallel:{N_GPU} GPUs ✅")

    fl_at=FocalLoss(weight=AT_W.to(DEVICE),gamma=2.0)
    fl_is=FocalLoss(weight=ISAT_W.to(DEVICE),gamma=2.0)
    try:
        groups=layerwise_groups(model,cfg["lr"],cfg["decay"])
        opt=torch.optim.AdamW(groups,weight_decay=WD)
        print(f"  Layer-wise LR: {len(groups)} groups ✅")
    except Exception as e:
        opt=torch.optim.AdamW(model.parameters(),lr=cfg["lr"],weight_decay=WD)
        print(f"  Flat LR: {e}")

    steps=math.ceil(len(TL)/cfg["accum"])*cfg["ep"]
    warmup=int(steps*cfg["wu"])
    sch=get_cosine_schedule_with_warmup(opt,warmup,steps)
    scaler=torch.amp.GradScaler("cuda"); opt.zero_grad(set_to_none=True)
    print(f"  Steps/ep:{math.ceil(len(TL)/cfg['accum'])} | Total:{steps} | Warmup:{warmup}\n")

    best_mr=0.0; no_imp=0; best_state=None; zero_loss_streak=0

    for ep in range(1,cfg["ep"]+1):
        model.train(); ep_loss=0.0; nb=0
        for bi,batch in enumerate(tqdm(TL,desc=f"  Ep{ep:02d} TRAIN",leave=False)):
            ids=batch["input_ids"].to(DEVICE); mask=batch["attention_mask"].to(DEVICE)
            al=batch["at_label"].to(DEVICE);   il=batch["isat_label"].to(DEVICE)
            with torch.amp.autocast("cuda"):
                ao,io=model(ids,mask)
                loss=(fl_at(ao,al)+fl_is(io,il))/cfg["accum"]
            scaler.scale(loss).backward()
            if (bi+1)%cfg["accum"]==0 or (bi+1)==len(TL):
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(),cfg["clip"])
                scaler.step(opt); scaler.update(); sch.step(); opt.zero_grad(set_to_none=True)
            if not torch.isnan(loss): ep_loss+=loss.item()*cfg["accum"]; nb+=1

        avg=round(ep_loss/max(nb,1),4)
        if avg==0.0:
            zero_loss_streak+=1; print(f"  ⚠ Loss=0.0 ({zero_loss_streak}/2)")
            if zero_loss_streak>=2: print(f"  ❌ Dead. Aborting {model_key}."); break
        else: zero_loss_streak=0

        model.eval(); at_t,at_p,is_t,is_p=[],[],[],[]
        with torch.no_grad():
            for batch in tqdm(DL,desc=f"  Ep{ep:02d} EVAL ",leave=False):
                with torch.amp.autocast("cuda"):
                    ao,io=model(batch["input_ids"].to(DEVICE),batch["attention_mask"].to(DEVICE))
                at_p.extend(ao.argmax(1).cpu().tolist()); at_t.extend(batch["at_label"].tolist())
                is_p.extend(io.argmax(1).cpu().tolist()); is_t.extend(batch["isat_label"].tolist())

        mr,mr_at,mr_is=macro_recall(at_t,at_p,is_t,is_p)
        print(f"\n  {'─'*60}")
        print(f"  Ep{ep:02d}|Loss:{avg:.4f}|MR:{mr:.4f}(at={mr_at:.4f},isAt={mr_is:.4f})|{round((time.time()-t0)/60,1)}min")
        print(f"  {'─'*60}")

        if mr>best_mr:
            best_mr=mr; no_imp=0
            raw=model.module if hasattr(model,"module") else model
            best_state=copy.deepcopy(raw.state_dict()); print(f"  ✅ Best MR={mr:.4f} saved")
        else:
            no_imp+=1; print(f"  ↘ No improvement ({no_imp}/{cfg['pat']})")
            if no_imp>=cfg["pat"]: print(f"  ⏹ Early stop ep{ep}"); break
        if ep%5==0:
            print(classification_report(at_t,at_p,target_names=AT_NAMES,zero_division=0))
            print(classification_report(is_t,is_p,target_names=ISAT_NAMES,zero_division=0))

    sd=OUT_DIR/model_key; sd.mkdir(exist_ok=True)
    if best_state: torch.save(best_state,sd/"best_model.pt"); tok.save_pretrained(sd)
    total=round((time.time()-t0)/60,1)
    print(f"\n  💾 {model_key.upper()} | Best MR:{best_mr:.4f} | {total}min")
    del model,opt,sch,scaler,fl_at,fl_is,TL,DL,tr_ds,dv_ds
    gc.collect(); torch.cuda.empty_cache(); torch.cuda.synchronize()
    print(f"  GPU free:{torch.cuda.mem_get_info()[0]/1e9:.2f} GB\n")
    return {"key":model_key,"best_mr":best_mr,"best_state":best_state,
            "tok":tok,"tok_name":cfg["name"],"cfg":cfg,"alive":best_state is not None}

# ── MAIN ──────────────────────────────────────────────────────────────────────
print("\n"+"█"*64)
print("  MHIPEX v10 | hmBERT + mBERT (stable WordPiece ensemble)")
print("  ⏱  Est. ~1.5h on T4 x2  |  🎯 Target MR: 0.56–0.62")
print("█"*64)

results={}
for key in MODEL_CFG: results[key]=train_model(key)

print("\n"+"="*64+"\n  INDIVIDUAL RESULTS\n"+"="*64)
for k,r in results.items():
    flag="✅" if r["alive"] and r["best_mr"]>=0.45 else "❌"
    print(f"  {k.upper():12s} | MR:{r['best_mr']:.4f}  {'█'*int(r['best_mr']*50)}  {flag}")

# ── Safe ensemble ─────────────────────────────────────────────────────────────
print("\n"+"="*64+"\n  SAFE ENSEMBLE (skip models MR<0.45)\n"+"="*64)
gc.collect(); torch.cuda.empty_cache()

all_probs={}; at_true_all=[]; is_true_all=[]
for key,res in results.items():
    if not res["alive"] or res["best_mr"]<0.45:
        print(f"  [{key}] SKIPPED (MR={res['best_mr']:.4f})"); continue
    cfg=res["cfg"]; dv_ds=HIPEDataset(PROC_DIR/"dev.jsonl",res["tok"],cfg["maxlen"])
    DL=DataLoader(dv_ds,batch_size=cfg["bs"],shuffle=False,num_workers=2,pin_memory=True)
    m=MHIPEXModel(res["tok_name"],n_new_tok=len(SPECIAL),dropout=cfg["drop"],grad_ckpt=cfg["ckpt"]).to(DEVICE)
    m.load_state_dict(res["best_state"]); m.eval()
    atp,isp,tmp_at,tmp_is=[],[],[],[]
    with torch.no_grad():
        for b in tqdm(DL,desc=f"  [{key}] inference",leave=False):
            with torch.amp.autocast("cuda"):
                ao,io=m(b["input_ids"].to(DEVICE),b["attention_mask"].to(DEVICE))
            atp.append(F.softmax(ao,dim=1).cpu().numpy())
            isp.append(F.softmax(io,dim=1).cpu().numpy())
            tmp_at.extend(b["at_label"].tolist()); tmp_is.extend(b["isat_label"].tolist())
    all_probs[key]={"at":np.concatenate(atp),"isat":np.concatenate(isp)}
    if not at_true_all: at_true_all=tmp_at; is_true_all=tmp_is
    del m; gc.collect(); torch.cuda.empty_cache()

good=list(all_probs.keys()); print(f"\n  Ensemble members: {good}")
if not good: print("❌ No models. Exiting."); exit()
at_ens  =sum(all_probs[k]["at"]   for k in good)/len(good)
isat_ens=sum(all_probs[k]["isat"] for k in good)/len(good)
at_pred =at_ens.argmax(axis=1).tolist()
isat_pred=isat_ens.argmax(axis=1).tolist()
mr_ens,_,_=macro_recall(at_true_all,at_pred,is_true_all,isat_pred)

print("\n  [at]:")
print(classification_report(at_true_all,at_pred,target_names=AT_NAMES,zero_division=0))
print("  [isAt]:")
print(classification_report(is_true_all,isat_pred,target_names=ISAT_NAMES,zero_division=0))

print("\n"+"█"*64+"\n  MHIPEX v10 — FINAL SCORECARD\n"+"█"*64)
for k,r in results.items(): print(f"  {k.upper():12s}: {r['best_mr']:.4f}")
print(f"  ENSEMBLE:     {mr_ens:.4f}  ← submit this")
print(f"  {'─'*42}")
print(f"  v8 best hmBERT: 0.5382  |  v9: 0.5347  |  SOTA: 0.6900")
gain=mr_ens-0.5382
print(f"  Gain over v8:  {'+' if gain>=0 else ''}{gain:.4f}")
if   mr_ens>=0.69: print("  🏆 BEAT SOTA!")
elif mr_ens>=0.60: print("  🥇 Excellent — well above v8!")
elif mr_ens>=0.55: print("  ✅ Clear improvement over v8!")
elif mr_ens>=0.54: print("  📈 Small gain")
else:              print("  ⚠  Flat vs v8 — see per-language breakdown")
print("█"*64)

dev_recs=[json.loads(l) for l in open(PROC_DIR/"dev.jsonl",encoding="utf-8")]
pred_path=OUT_DIR/"ensemble_predictions_v10.jsonl"
with open(pred_path,"w",encoding="utf-8") as f:
    for i,rec in enumerate(dev_recs):
        f.write(json.dumps({"lang":rec["lang"],
            "at_true":AT_NAMES[at_true_all[i]],"at_pred":AT_NAMES[at_pred[i]],
            "isat_true":ISAT_NAMES[is_true_all[i]],"isat_pred":ISAT_NAMES[isat_pred[i]],
            "at_correct":at_pred[i]==at_true_all[i],
            "isat_correct":isat_pred[i]==is_true_all[i]},ensure_ascii=False)+"\n")

lang_stats={}
for row in [json.loads(l) for l in open(pred_path,encoding="utf-8")]:
    lg=row["lang"]
    if lg not in lang_stats: lang_stats[lg]={"at_ok":0,"n":0,"is_ok":0}
    lang_stats[lg]["at_ok"]+=int(row["at_correct"])
    lang_stats[lg]["is_ok"]+=int(row["isat_correct"])
    lang_stats[lg]["n"]+=1

print("\n  PER-LANGUAGE")
print(f"  {'Lang':6s}  {'at acc':>8s}  {'isAt acc':>10s}  {'N':>6s}")
print(f"  {'─'*36}")
for lg,s in sorted(lang_stats.items()):
    print(f"  {lg:6s}  {s['at_ok']/s['n']:8.3f}  {s['is_ok']/s['n']:10.3f}  {s['n']:6,}")
print(f"\n✅ {pred_path}\n✅ out/hmbert/  |  out/mbert/")
print(f"\n── DONE. Submit ensemble_predictions_v10.jsonl ──\n")


✅ Packages ready
Device : cuda | GPUs : 2 | CUDA : 12.8

── Data ─────────────────────────────────────────────────────────────
  en-train.jsonl (cached)
  fr-train.jsonl (cached)
  de-train.jsonl (cached)
  en-dev.jsonl (cached)
  fr-dev.jsonl (cached)
  de-dev.jsonl (cached)
✅ Data ready

── Dataset ──────────────────────────────────────────────────────────
  en train: 496
  fr train: 4,450
  de train: 1,224
✅ train: 6,170

  en dev  : 151
  fr dev  : 1,498
  de dev  : 432
✅ dev: 2,081


████████████████████████████████████████████████████████████████
  MHIPEX v10 | hmBERT + mBERT (stable WordPiece ensemble)
  ⏱  Est. ~1.5h on T4 x2  |  🎯 Target MR: 0.56–0.62
████████████████████████████████████████████████████████████████

  HMBERT | dbmdz/bert-base-historic-multilingual-cased
  LR=8e-06 | EffBS=64 | Pat=8 | clip=0.5

  GPU free: 15.53 GB
    '<P>'→[32000] (✅)
    '</P>'→[32001] (✅)
    '<L>'→[32002] (✅)
    '</L>'→[32003] (✅)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


  DataParallel:2 GPUs ✅
  Layer-wise LR: 16 groups ✅
  Steps/ep:97 | Total:3395 | Warmup:407




  ────────────────────────────────────────────────────────────
  Ep01|Loss:1.4127|MR:0.4167(at=0.3333,isAt=0.5000)|1.3min
  ────────────────────────────────────────────────────────────
  ✅ Best MR=0.4167 saved



  ────────────────────────────────────────────────────────────
  Ep02|Loss:1.2853|MR:0.4167(at=0.3333,isAt=0.5000)|2.4min
  ────────────────────────────────────────────────────────────
  ↘ No improvement (1/8)


...


  ────────────────────────────────────────────────────────────
  Ep33|Loss:0.4334|MR:0.5270(at=0.4287,isAt=0.6253)|39.5min
  ────────────────────────────────────────────────────────────
  ↘ No improvement (1/8)



  ────────────────────────────────────────────────────────────
  Ep34|Loss:0.4238|MR:0.5276(at=0.4286,isAt=0.6266)|40.6min
  ────────────────────────────────────────────────────────────
  ↘ No improvement (2/8)



  ────────────────────────────────────────────────────────────
  Ep35|Loss:0.4307|MR:0.5266(at=0.4289,isAt=0.6243)|41.8min
  ────────────────────────────────────────────────────────────
  ↘ No improvement (3/8)
              precision    recall  f1-score   support

       FALSE       0.74      0.11      0.19      1264
    PROBABLE       0.30      0.59      0.40       568
        TRUE       0.19      0.59      0.29       249

    accuracy                           0.30      2081
   macro avg       0.41      0.43      0.29      2081
weighted avg       0.55      0.30      0.26      2081

              precision    recall  f1-score   support

       FALSE       0.95      0.69      0.79      1907
        TRUE       0.14      0.56      0.22       174

    accuracy                           0.68      2081
   macro avg       0.54      0.62      0.51      2081
weighted avg       0.88      0.68      0.75      2081


  💾 HMBERT | Best MR:0.5347 | 41.9min
  GPU free:13.95 GB


  MBERT | google/be

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


OSError: google/bert-base-multilingual-cased is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `huggingface-cli login` or by passing `token=<your_token>`

In [2]:
# ══════════════════════════════════════════════════════════════════
#  MHIPEX v10 — CONTINUATION CELL
#  Run this in the NEXT cell after hmBERT finished.
#  mBERT uses "bert-base-multilingual-cased" (no google/ prefix → no 401)
#  Then soft-ensembles hmBERT + mBERT → submit
# ══════════════════════════════════════════════════════════════════

import os, math, json, re, random, copy, gc, time
os.environ["PYTORCH_ALLOC_CONF"]     = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from pathlib import Path
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from transformers import AutoModel, AutoTokenizer, get_cosine_schedule_with_warmup
from sklearn.metrics import recall_score, classification_report
from tqdm import tqdm

SEED = 42
def set_seed(s=SEED):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
set_seed()
torch.backends.cuda.matmul.allow_tf32 = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_GPU  = torch.cuda.device_count()
print(f"Device: {DEVICE} | GPUs: {N_GPU}")

# ── Constants ─────────────────────────────────────────────────────────────────
AT_MAP     = {"FALSE":0, "PROBABLE":1, "TRUE":2}
ISAT_MAP   = {"FALSE":0, "TRUE":1}
AT_NAMES   = ["FALSE", "PROBABLE", "TRUE"]
ISAT_NAMES = ["FALSE", "TRUE"]
SPECIAL    = ["<P>", "</P>", "<L>", "</L>"]
AT_W       = torch.tensor([0.80, 1.50, 2.40], dtype=torch.float32)
ISAT_W     = torch.tensor([0.70, 2.60],        dtype=torch.float32)
WD         = 0.01

PROC_DIR = Path("proc")
OUT_DIR  = Path("out"); OUT_DIR.mkdir(exist_ok=True)

# ── mBERT Config ─────────────────────────────────────────────────────────────
# KEY: "bert-base-multilingual-cased" — no google/ prefix → no 401 error
MBERT_CFG = {
    "name":    "bert-base-multilingual-cased",
    "bs":32, "accum":2, "lr":8e-6, "decay":0.90,
    "ckpt":False, "pat":8, "ep":35, "wu":0.12, "maxlen":256, "drop":0.15, "clip":0.5,
}

# ── HMBERT result from previous cell ─────────────────────────────────────────
HMBERT_MR   = 0.5347   # update if your run gave different
HMBERT_DIR  = OUT_DIR / "hmbert"

# ── Dataset ───────────────────────────────────────────────────────────────────
class HIPEDataset(Dataset):
    def __init__(self, path, tok, maxlen=256):
        self.data = [json.loads(l) for l in open(path, encoding="utf-8")]
        self.tok = tok; self.maxlen = maxlen
    def __len__(self): return len(self.data)
    def __getitem__(self, i):
        d = self.data[i]
        enc = self.tok(d["text"], max_length=self.maxlen, truncation=True,
                       padding="max_length", return_tensors="pt")
        return {"input_ids":      enc["input_ids"].squeeze(0),
                "attention_mask": enc["attention_mask"].squeeze(0),
                "at_label":       torch.tensor(d["at"],   dtype=torch.long),
                "isat_label":     torch.tensor(d["isat"], dtype=torch.long)}

def balanced_sampler(ds):
    labels = np.array([r["at"] for r in ds.data])
    counts = np.bincount(labels, minlength=3).astype(float)
    return WeightedRandomSampler([1.0/counts[l] for l in labels], len(labels), replacement=True)

# ── Focal Loss ────────────────────────────────────────────────────────────────
class FocalLoss(nn.Module):
    def __init__(self, weight=None, gamma=2.0):
        super().__init__(); self.gamma = gamma
        self.register_buffer("weight", weight)
    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.weight, reduction="none")
        return (((1 - torch.exp(-ce)) ** self.gamma) * ce).mean()

# ── Model ─────────────────────────────────────────────────────────────────────
class MHIPEXModel(nn.Module):
    def __init__(self, name, n_new_tok, dropout=0.15, grad_ckpt=False):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(name)
        self.encoder.resize_token_embeddings(self.encoder.config.vocab_size + n_new_tok)
        if grad_ckpt: self.encoder.gradient_checkpointing_enable()
        h = self.encoder.config.hidden_size
        self.layer_norm = nn.LayerNorm(h); self.dropout = nn.Dropout(dropout)
        self.head_at   = nn.Sequential(nn.Linear(h,h//2), nn.GELU(), nn.Dropout(dropout), nn.Linear(h//2,3))
        self.head_isat = nn.Sequential(nn.Linear(h,h//2), nn.GELU(), nn.Dropout(dropout), nn.Linear(h//2,2))
    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = self.layer_norm(self.dropout(out.last_hidden_state[:, 0, :]))
        return self.head_at(cls), self.head_isat(cls)

def layerwise_groups(model, base_lr, decay):
    raw = model.module if hasattr(model, "module") else model
    enc = raw.encoder; groups = []
    layers = getattr(getattr(enc, "encoder", enc), "layer", None)
    if layers:
        n = len(layers)
        for i, layer in enumerate(layers):
            groups.append({"params": list(layer.parameters()), "lr": base_lr*(decay**(n-1-i))})
    if hasattr(enc, "embeddings"):
        groups.append({"params": list(enc.embeddings.parameters()),
                       "lr": base_lr*(decay**(len(layers) if layers else 12))})
    for head in [raw.head_at, raw.head_isat, raw.layer_norm]:
        groups.append({"params": list(head.parameters()), "lr": base_lr})
    return groups

def macro_recall(at_t, at_p, is_t, is_p):
    r1 = recall_score(at_t, at_p, average="macro", zero_division=0)
    r2 = recall_score(is_t, is_p, average="macro", zero_division=0)
    return round((r1+r2)/2, 4), round(r1, 4), round(r2, 4)

# ════════════════════════════════════════════════════════════════════════════
#  TRAIN mBERT
# ════════════════════════════════════════════════════════════════════════════
cfg = MBERT_CFG; eff_bs = cfg["bs"]*cfg["accum"]
print(f"\n{'='*64}")
print(f"  mBERT | {cfg['name']}")
print(f"  LR={cfg['lr']} | EffBS={eff_bs} | Pat={cfg['pat']} | clip={cfg['clip']}")
print(f"{'='*64}\n")
gc.collect(); torch.cuda.empty_cache(); torch.cuda.synchronize()
print(f"  GPU free: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")
set_seed(); t0 = time.time()

tok = AutoTokenizer.from_pretrained(cfg["name"])
n   = tok.add_special_tokens({"additional_special_tokens": SPECIAL})
for st in SPECIAL:
    ids = tok.encode(st, add_special_tokens=False)
    print(f"    '{st}' → {ids} ({'✅' if len(ids)==1 else '⚠ split'})")

tr_ds = HIPEDataset(PROC_DIR/"train.jsonl", tok, cfg["maxlen"])
dv_ds = HIPEDataset(PROC_DIR/"dev.jsonl",   tok, cfg["maxlen"])
TL = DataLoader(tr_ds, batch_size=cfg["bs"], sampler=balanced_sampler(tr_ds), num_workers=2, pin_memory=True)
DL = DataLoader(dv_ds, batch_size=cfg["bs"], shuffle=False, num_workers=2, pin_memory=True)

model = MHIPEXModel(cfg["name"], n_new_tok=n, dropout=cfg["drop"], grad_ckpt=cfg["ckpt"]).to(DEVICE)
if N_GPU > 1: model = nn.DataParallel(model); print(f"  DataParallel:{N_GPU} GPUs ✅")

fl_at   = FocalLoss(weight=AT_W.to(DEVICE),   gamma=2.0)
fl_isat = FocalLoss(weight=ISAT_W.to(DEVICE), gamma=2.0)

try:
    groups = layerwise_groups(model, cfg["lr"], cfg["decay"])
    opt    = torch.optim.AdamW(groups, weight_decay=WD)
    print(f"  Layer-wise LR: {len(groups)} groups ✅")
except Exception as e:
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=WD)
    print(f"  Flat LR fallback ({e})")

steps  = math.ceil(len(TL)/cfg["accum"]) * cfg["ep"]
warmup = int(steps * cfg["wu"])
sch    = get_cosine_schedule_with_warmup(opt, warmup, steps)
scaler = torch.amp.GradScaler("cuda")
opt.zero_grad(set_to_none=True)
print(f"  Steps/ep:{math.ceil(len(TL)/cfg['accum'])} | Total:{steps} | Warmup:{warmup}\n")

best_mr=0.0; no_imp=0; best_state=None; zero_streak=0

for ep in range(1, cfg["ep"]+1):
    model.train(); ep_loss=0.0; nb=0
    for bi, batch in enumerate(tqdm(TL, desc=f"  Ep{ep:02d} TRAIN", leave=False)):
        ids  = batch["input_ids"].to(DEVICE);   mask = batch["attention_mask"].to(DEVICE)
        al   = batch["at_label"].to(DEVICE);    il   = batch["isat_label"].to(DEVICE)
        with torch.amp.autocast("cuda"):
            ao, io = model(ids, mask)
            loss   = (fl_at(ao, al) + fl_isat(io, il)) / cfg["accum"]
        scaler.scale(loss).backward()
        if (bi+1) % cfg["accum"] == 0 or (bi+1) == len(TL):
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg["clip"])
            scaler.step(opt); scaler.update(); sch.step(); opt.zero_grad(set_to_none=True)
        if not torch.isnan(loss): ep_loss += loss.item()*cfg["accum"]; nb += 1

    avg = round(ep_loss/max(nb,1), 4)
    if avg == 0.0:
        zero_streak += 1; print(f"  ⚠  Loss=0.0 ({zero_streak}/2)")
        if zero_streak >= 2: print(f"  ❌ Model dead. Aborting."); break
    else:
        zero_streak = 0

    model.eval(); at_t,at_p,is_t,is_p=[],[],[],[]
    with torch.no_grad():
        for batch in tqdm(DL, desc=f"  Ep{ep:02d} EVAL ", leave=False):
            with torch.amp.autocast("cuda"):
                ao, io = model(batch["input_ids"].to(DEVICE), batch["attention_mask"].to(DEVICE))
            at_p.extend(ao.argmax(1).cpu().tolist()); at_t.extend(batch["at_label"].tolist())
            is_p.extend(io.argmax(1).cpu().tolist()); is_t.extend(batch["isat_label"].tolist())

    mr, mr_at, mr_is = macro_recall(at_t, at_p, is_t, is_p)
    print(f"\n  {'─'*60}")
    print(f"  Ep{ep:02d}|Loss:{avg:.4f}|MR:{mr:.4f}(at={mr_at:.4f},isAt={mr_is:.4f})|{round((time.time()-t0)/60,1)}min")
    print(f"  {'─'*60}")

    if mr > best_mr:
        best_mr=mr; no_imp=0
        raw = model.module if hasattr(model, "module") else model
        best_state = copy.deepcopy(raw.state_dict()); print(f"  ✅ Best MR={mr:.4f} saved")
    else:
        no_imp += 1; print(f"  ↘ No improvement ({no_imp}/{cfg['pat']})")
        if no_imp >= cfg["pat"]: print(f"  ⏹ Early stop ep{ep}"); break

    if ep % 5 == 0:
        print(classification_report(at_t, at_p, target_names=AT_NAMES, zero_division=0))
        print(classification_report(is_t, is_p, target_names=ISAT_NAMES, zero_division=0))

# Save mBERT
sd = OUT_DIR/"mbert"; sd.mkdir(exist_ok=True)
if best_state:
    raw = model.module if hasattr(model,"module") else model
    raw.load_state_dict(best_state)
    torch.save(best_state, sd/"best_model.pt"); tok.save_pretrained(sd)
mbert_alive = best_state is not None and best_mr >= 0.45
total = round((time.time()-t0)/60, 1)
print(f"\n  💾 mBERT | Best MR:{best_mr:.4f} | {total}min")
del model, opt, sch, scaler, fl_at, fl_isat, TL, DL, tr_ds, dv_ds
gc.collect(); torch.cuda.empty_cache(); torch.cuda.synchronize()
print(f"  GPU free:{torch.cuda.mem_get_info()[0]/1e9:.2f} GB")

# ════════════════════════════════════════════════════════════════════════════
#  SAFE ENSEMBLE — hmBERT + mBERT
# ════════════════════════════════════════════════════════════════════════════
print(f"\n{'='*64}\n  SAFE ENSEMBLE\n{'='*64}")
gc.collect(); torch.cuda.empty_cache()

def get_probs(model_dir, model_name, n_special=4, bs=32, maxlen=256):
    saved_tok = AutoTokenizer.from_pretrained(str(model_dir))
    saved_tok.add_special_tokens({"additional_special_tokens": SPECIAL})
    ds = HIPEDataset(PROC_DIR/"dev.jsonl", saved_tok, maxlen)
    dl = DataLoader(ds, batch_size=bs, shuffle=False, num_workers=2, pin_memory=True)
    m  = MHIPEXModel(model_name, n_new_tok=n_special, dropout=0.15).to(DEVICE)
    m.load_state_dict(torch.load(str(model_dir/"best_model.pt"), map_location=DEVICE))
    m.eval()
    atp, isp, at_t, is_t = [], [], [], []
    with torch.no_grad():
        for b in tqdm(dl, desc=f"  [{model_dir.name}] infer", leave=False):
            with torch.amp.autocast("cuda"):
                ao, io = m(b["input_ids"].to(DEVICE), b["attention_mask"].to(DEVICE))
            atp.append(F.softmax(ao, dim=1).cpu().numpy())
            isp.append(F.softmax(io, dim=1).cpu().numpy())
            at_t.extend(b["at_label"].tolist()); is_t.extend(b["isat_label"].tolist())
    del m; gc.collect(); torch.cuda.empty_cache()
    return np.concatenate(atp), np.concatenate(isp), at_t, is_t

all_at, all_is = [], []
at_true = []; is_true = []

# hmBERT
print("  Loading hmBERT...")
hat, his, at_t, is_t = get_probs(HMBERT_DIR, "dbmdz/bert-base-historic-multilingual-cased")
all_at.append(hat); all_is.append(his); at_true=at_t; is_true=is_t
print(f"  hmBERT probs loaded ✅")

# mBERT (only if alive)
if mbert_alive:
    print("  Loading mBERT...")
    mat, mis, _, _ = get_probs(OUT_DIR/"mbert", "bert-base-multilingual-cased")
    all_at.append(mat); all_is.append(mis)
    print(f"  mBERT probs loaded ✅")
else:
    print(f"  mBERT SKIPPED (MR={best_mr:.4f} < 0.45 or model dead)")

# Soft vote
n_models = len(all_at)
at_ens   = sum(all_at)   / n_models
isat_ens = sum(all_is)   / n_models
at_pred   = at_ens.argmax(axis=1).tolist()
isat_pred = isat_ens.argmax(axis=1).tolist()
mr_ens, mr_at_e, mr_is_e = macro_recall(at_true, at_pred, is_true, isat_pred)

print(f"\n  Models in ensemble: {n_models} ({'hmBERT + mBERT' if n_models==2 else 'hmBERT only'})")
print("\n  [at] Ensemble Report:")
print(classification_report(at_true, at_pred, target_names=AT_NAMES, zero_division=0))
print("  [isAt] Ensemble Report:")
print(classification_report(is_true, isat_pred, target_names=ISAT_NAMES, zero_division=0))

# ── Final Scorecard ───────────────────────────────────────────────────────────
print(f"\n{'█'*64}")
print(f"  MHIPEX v10 — FINAL SCORECARD")
print(f"{'█'*64}")
print(f"  hmBERT (this run): {HMBERT_MR:.4f}")
print(f"  mBERT  (this run): {best_mr:.4f}{'  ✅' if mbert_alive else '  ❌ skipped'}")
print(f"  ENSEMBLE (FINAL):  {mr_ens:.4f}  ← submit this")
print(f"  {'─'*42}")
print(f"  v8 best:   0.5382  |  v9 best: 0.5347  |  SOTA: 0.6900")
gain = mr_ens - 0.5382
print(f"  Gain over v8:  {'+' if gain>=0 else ''}{gain:.4f}")
if   mr_ens >= 0.69: print("  🏆 BEAT hmBERT SOTA!")
elif mr_ens >= 0.60: print("  🥇 Excellent — clear improvement!")
elif mr_ens >= 0.55: print("  ✅ Solid gain over v8!")
elif mr_ens >= 0.54: print("  📈 Small gain — mBERT helped")
else:                print("  ⚠  hmBERT alone is still your best — check if mBERT was alive")
print(f"{'█'*64}")

# Save predictions
dev_recs  = [json.loads(l) for l in open(PROC_DIR/"dev.jsonl", encoding="utf-8")]
pred_path = OUT_DIR/"ensemble_predictions_v10.jsonl"
with open(pred_path, "w", encoding="utf-8") as f:
    for i, rec in enumerate(dev_recs):
        f.write(json.dumps({"lang": rec["lang"],
            "at_true":   AT_NAMES[at_true[i]],   "at_pred":   AT_NAMES[at_pred[i]],
            "isat_true": ISAT_NAMES[is_true[i]],  "isat_pred": ISAT_NAMES[isat_pred[i]],
            "at_correct": at_pred[i]==at_true[i], "isat_correct": isat_pred[i]==is_true[i]},
            ensure_ascii=False)+"\n")

# Per-language
lang_stats = {}
for row in [json.loads(l) for l in open(pred_path, encoding="utf-8")]:
    lg = row["lang"]
    if lg not in lang_stats: lang_stats[lg]={"at_ok":0,"is_ok":0,"n":0}
    lang_stats[lg]["at_ok"] += int(row["at_correct"])
    lang_stats[lg]["is_ok"] += int(row["isat_correct"])
    lang_stats[lg]["n"]     += 1
print("\n  PER-LANGUAGE")
print(f"  {'Lang':6s}  {'at acc':>8s}  {'isAt acc':>10s}  {'N':>6s}")
print(f"  {'─'*36}")
for lg, s in sorted(lang_stats.items()):
    print(f"  {lg:6s}  {s['at_ok']/s['n']:8.3f}  {s['is_ok']/s['n']:10.3f}  {s['n']:6,}")

print(f"\n✅ Predictions → {pred_path}")
print(f"✅ Models      → out/hmbert/  |  out/mbert/")
print(f"\n── DONE. Submit ensemble_predictions_v10.jsonl ──\n")

Device: cuda | GPUs: 2

  mBERT | bert-base-multilingual-cased
  LR=8e-06 | EffBS=64 | Pat=8 | clip=0.5

  GPU free: 14.39 GB


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


  GPU free:12.99 GB

  SAFE ENSEMBLE
  Loading hmBERT...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


  hmBERT probs loaded ✅
  Loading mBERT...


  mBERT probs loaded ✅

  Models in ensemble: 2 (hmBERT + mBERT)

  [at] Ensemble Report:
              precision    recall  f1-score   support

       FALSE       0.72      0.06      0.11      1264
    PROBABLE       0.29      0.62      0.40       568
        TRUE       0.20      0.60      0.30       249

    accuracy                           0.28      2081
   macro avg       0.40      0.43      0.27      2081
weighted avg       0.54      0.28      0.21      2081

  [isAt] Ensemble Report:
              precision    recall  f1-score   support

       FALSE       0.95      0.69      0.80      1907
        TRUE       0.14      0.57      0.23       174

    accuracy                           0.68      2081
   macro avg       0.54      0.63      0.51      2081
weighted avg       0.88      0.68      0.75      2081


████████████████████████████████████████████████████████████████
  MHIPEX v10 — FINAL SCORECARD
████████████████████████████████████████████████████████████████
  hmBERT (this

In [4]:
# ══════════════════════════════════════════════════════════════════
#  TIER 1 — FIX 1: Logical Constraint Post-Processing (5 minutes)
#  Rule: if at=FALSE → force isAt=FALSE (task definition requires this)
#  Apply to BEST saved predictions (hmBERT 0.5382 from v8, or v9/v10)
#  Expected gain: +0.02 to +0.04 MR instantly, zero retraining
# ══════════════════════════════════════════════════════════════════

import json
from pathlib import Path
from sklearn.metrics import recall_score

AT_NAMES   = ["FALSE", "PROBABLE", "TRUE"]
ISAT_NAMES = ["FALSE", "TRUE"]

OUT_DIR = Path("out")

# ── Load your best predictions ─────────────────────────────────────────────
# Priority: v8 > v9 > v10 (pick whichever exists and has highest MR)
for fname in ["ensemble_predictions_v8.jsonl",
              "ensemble_predictions_v9.jsonl",
              "ensemble_predictions_v10.jsonl"]:
    p = OUT_DIR / fname
    if p.exists():
        PRED_FILE = p
        print(f"  Using: {fname}")
        break

recs = [json.loads(l) for l in open(PRED_FILE, encoding="utf-8")]

# ── Before fix ─────────────────────────────────────────────────────────────
at_t  = [AT_NAMES.index(r["at_true"])   for r in recs]
at_p  = [AT_NAMES.index(r["at_pred"])   for r in recs]
is_t  = [ISAT_NAMES.index(r["isat_true"]) for r in recs]
is_p  = [ISAT_NAMES.index(r["isat_pred"]) for r in recs]

mr_before = round((
    recall_score(at_t, at_p, average="macro", zero_division=0) +
    recall_score(is_t, is_p, average="macro", zero_division=0)
) / 2, 4)
print(f"\n  MR BEFORE fix: {mr_before:.4f}")

# ── Apply logical constraint ────────────────────────────────────────────────
# HIPE-2026 task definition: at=FALSE means person was NOT at that location
# Therefore isAt (around publication time) MUST also be FALSE
fixed = 0
is_p_fixed = []
for i, r in enumerate(recs):
    if r["at_pred"] == "FALSE":
        is_p_fixed.append(ISAT_NAMES.index("FALSE"))
        if r["isat_pred"] != "FALSE":
            fixed += 1
    else:
        is_p_fixed.append(ISAT_NAMES.index(r["isat_pred"]))

mr_after = round((
    recall_score(at_t, at_p, average="macro", zero_division=0) +
    recall_score(is_t, is_p_fixed, average="macro", zero_division=0)
) / 2, 4)

print(f"  Predictions corrected: {fixed} / {len(recs)}")
print(f"  MR AFTER  fix: {mr_after:.4f}")
print(f"  GAIN:         +{mr_after - mr_before:.4f}")

# ── Save fixed predictions ──────────────────────────────────────────────────
fixed_path = OUT_DIR / "predictions_tier1_constraint.jsonl"
with open(fixed_path, "w", encoding="utf-8") as f:
    for i, r in enumerate(recs):
        r["isat_pred_orig"] = r["isat_pred"]
        r["isat_pred"]      = ISAT_NAMES[is_p_fixed[i]]
        r["isat_correct"]   = (r["isat_pred"] == r["isat_true"])
        r["at_correct"]     = (r["at_pred"]   == r["at_true"])
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print(f"\n✅ Saved → {fixed_path}")
print(f"  Submit this instead of the original predictions!\n")


  Using: ensemble_predictions_v8.jsonl

  MR BEFORE fix: 0.5112
  Predictions corrected: 2 / 2081
  MR AFTER  fix: 0.5114
  GAIN:         +0.0002

✅ Saved → out/predictions_tier1_constraint.jsonl
  Submit this instead of the original predictions!



In [5]:
# ══════════════════════════════════════════════════════════════════
#  TIER 1 — FIX 2: Threshold Calibration (30 minutes, no retraining)
#  Grid-search optimal decision thresholds on dev set per class
#  Uses saved hmBERT model probabilities directly
#  Expected gain: +0.03 to +0.06 MR on top of Fix 1
# ══════════════════════════════════════════════════════════════════

import json, gc
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer
from sklearn.metrics import recall_score
from itertools import product
from tqdm import tqdm

DEVICE   = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AT_NAMES   = ["FALSE", "PROBABLE", "TRUE"]
ISAT_NAMES = ["FALSE", "TRUE"]
SPECIAL    = ["<P>", "</P>", "<L>", "</L>"]
PROC_DIR   = Path("proc")
OUT_DIR    = Path("out")

class HIPEDataset(Dataset):
    def __init__(self, path, tok, maxlen=256):
        self.data = [json.loads(l) for l in open(path, encoding="utf-8")]
        self.tok = tok; self.maxlen = maxlen
    def __len__(self): return len(self.data)
    def __getitem__(self, i):
        d = self.data[i]
        enc = self.tok(d["text"], max_length=self.maxlen, truncation=True,
                       padding="max_length", return_tensors="pt")
        return {"input_ids": enc["input_ids"].squeeze(0),
                "attention_mask": enc["attention_mask"].squeeze(0),
                "at_label":   torch.tensor(d["at"],   dtype=torch.long),
                "isat_label": torch.tensor(d["isat"], dtype=torch.long)}

class MHIPEXModel(nn.Module):
    def __init__(self, name, n_new_tok, dropout=0.15):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(name)
        self.encoder.resize_token_embeddings(self.encoder.config.vocab_size + n_new_tok)
        h = self.encoder.config.hidden_size
        self.layer_norm = nn.LayerNorm(h); self.dropout = nn.Dropout(dropout)
        self.head_at   = nn.Sequential(nn.Linear(h,h//2), nn.GELU(), nn.Dropout(dropout), nn.Linear(h//2,3))
        self.head_isat = nn.Sequential(nn.Linear(h,h//2), nn.GELU(), nn.Dropout(dropout), nn.Linear(h//2,2))
    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = self.layer_norm(self.dropout(out.last_hidden_state[:, 0, :]))
        return self.head_at(cls), self.head_isat(cls)

# ── Get raw probabilities from hmBERT ─────────────────────────────────────
print("  Loading hmBERT for probability extraction...")
MODEL_DIR = OUT_DIR / "hmbert"
MODEL_NAME = "dbmdz/bert-base-historic-multilingual-cased"

tok = AutoTokenizer.from_pretrained(str(MODEL_DIR))
tok.add_special_tokens({"additional_special_tokens": SPECIAL})

ds = HIPEDataset(PROC_DIR/"dev.jsonl", tok, 256)
dl = DataLoader(ds, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

model = MHIPEXModel(MODEL_NAME, n_new_tok=len(SPECIAL)).to(DEVICE)
state = torch.load(str(MODEL_DIR/"best_model.pt"), map_location=DEVICE)
model.load_state_dict(state); model.eval()

at_probs_all, is_probs_all, at_true, is_true = [], [], [], []
with torch.no_grad():
    for b in tqdm(dl, desc="  Extracting probs"):
        with torch.amp.autocast("cuda"):
            ao, io = model(b["input_ids"].to(DEVICE), b["attention_mask"].to(DEVICE))
        at_probs_all.append(F.softmax(ao, dim=1).cpu().numpy())
        is_probs_all.append(F.softmax(io, dim=1).cpu().numpy())
        at_true.extend(b["at_label"].tolist())
        is_true.extend(b["isat_label"].tolist())

at_probs = np.concatenate(at_probs_all)   # [N, 3]
is_probs = np.concatenate(is_probs_all)   # [N, 2]
del model; gc.collect(); torch.cuda.empty_cache()
print(f"  Probs extracted: {at_probs.shape}, {is_probs.shape}")

# ── Baseline (argmax) ────────────────────────────────────────────────────
at_pred_base  = at_probs.argmax(axis=1)
is_pred_base  = is_probs.argmax(axis=1)
mr_base = round((
    recall_score(at_true, at_pred_base, average="macro", zero_division=0) +
    recall_score(is_true, is_pred_base, average="macro", zero_division=0)
) / 2, 4)
print(f"\n  Baseline argmax MR: {mr_base:.4f}")

# ── Grid search: at thresholds ─────────────────────────────────────────────
# For 3-class: predict class c if prob[c] > threshold[c], else fallback to argmax
# Sweep PROBABLE and TRUE thresholds (FALSE gets the remainder)
print("\n  Grid searching at-thresholds...")
best_mr = mr_base; best_t_at = None

thresholds = [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45]
for t_prob, t_true in product(thresholds, thresholds):
    at_pred_t = []
    for p in at_probs:
        if   p[2] >= t_true: at_pred_t.append(2)   # TRUE
        elif p[1] >= t_prob: at_pred_t.append(1)   # PROBABLE
        else:                at_pred_t.append(0)   # FALSE
    mr_t = recall_score(at_true, at_pred_t, average="macro", zero_division=0)
    if mr_t > recall_score(at_true, at_pred_base, average="macro", zero_division=0):
        best_t_at = (t_prob, t_true)

# ── Grid search: isAt threshold ───────────────────────────────────────────
print("  Grid searching isAt-threshold...")
best_t_is = 0.5
for t_is in [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]:
    is_pred_t = (is_probs[:, 1] >= t_is).astype(int)
    mr_t = recall_score(is_true, is_pred_t, average="macro", zero_division=0)
    if mr_t > recall_score(is_true, is_pred_base, average="macro", zero_division=0):
        best_t_is = t_is

print(f"  Best at-thresholds:   PROBABLE>={best_t_at[0] if best_t_at else 'default'}, TRUE>={best_t_at[1] if best_t_at else 'default'}")
print(f"  Best isAt-threshold:  TRUE>={best_t_is:.2f}")

# ── Apply best thresholds + logical constraint ───────────────────────────
at_pred_final = []
for p in at_probs:
    if best_t_at:
        if   p[2] >= best_t_at[1]: at_pred_final.append(2)
        elif p[1] >= best_t_at[0]: at_pred_final.append(1)
        else:                      at_pred_final.append(0)
    else:
        at_pred_final.append(int(p.argmax()))

is_pred_final = []
for i, p in enumerate(is_probs):
    # Logical constraint: if at=FALSE force isAt=FALSE
    if at_pred_final[i] == 0:
        is_pred_final.append(0)
    else:
        is_pred_final.append(int(p[1] >= best_t_is))

mr_final = round((
    recall_score(at_true, at_pred_final, average="macro", zero_division=0) +
    recall_score(is_true, is_pred_final, average="macro", zero_division=0)
) / 2, 4)

print(f"\n{'='*56}")
print(f"  Baseline (argmax):          MR = {mr_base:.4f}")
print(f"  + Threshold calibration:    MR = {mr_final:.4f}")
print(f"  Gain:                      +{mr_final - mr_base:.4f}")
print(f"  v8 best (submitted):        MR = 0.5382")
print(f"  NEW BEST:                   MR = {mr_final:.4f}  {'🏆 NEW BEST!' if mr_final > 0.5382 else ''}")
print(f"{'='*56}")

# ── Save ─────────────────────────────────────────────────────────────────
dev_recs = [json.loads(l) for l in open(PROC_DIR/"dev.jsonl", encoding="utf-8")]
out_path = OUT_DIR / "predictions_tier1_calibrated.jsonl"
with open(out_path, "w", encoding="utf-8") as f:
    for i, rec in enumerate(dev_recs):
        f.write(json.dumps({
            "lang": rec["lang"],
            "at_true":    AT_NAMES[at_true[i]],
            "at_pred":    AT_NAMES[at_pred_final[i]],
            "isat_true":  ISAT_NAMES[is_true[i]],
            "isat_pred":  ISAT_NAMES[is_pred_final[i]],
            "at_correct": at_pred_final[i]==at_true[i],
            "isat_correct": is_pred_final[i]==is_true[i]
        }, ensure_ascii=False) + "\n")

print(f"\n✅ Submit → {out_path}")


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


  Loading hmBERT for probability extraction...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
  Extracting probs: 100%|██████████| 66/66 [00:09<00:00,  6.80it/s]


  Probs extracted: (2081, 3), (2081, 2)

  Baseline argmax MR: 0.5347

  Grid searching at-thresholds...
  Grid searching isAt-threshold...
  Best at-thresholds:   PROBABLE>=default, TRUE>=default
  Best isAt-threshold:  TRUE>=0.50

  Baseline (argmax):          MR = 0.5347
  + Threshold calibration:    MR = 0.5366
  Gain:                      +0.0019
  v8 best (submitted):        MR = 0.5382
  NEW BEST:                   MR = 0.5366  

✅ Submit → out/predictions_tier1_calibrated.jsonl


In [6]:
# ══════════════════════════════════════════════════════════════════
#  TIER 1 — FIX 3: Add DATE + LANG tokens (Retrain v11)
#  Enriches input from:
#    <P> person </P> <L> location </L> text
#  To:
#    <P> person </P> <L> location </L> <DATE> 1923 </DATE> <LANG> fr </LANG> text
#  Expected gain: +0.02–0.04 MR (temporal signals critical for isAt)
#  Run this AFTER Fixes 1 and 2 — this requires a full retrain
# ══════════════════════════════════════════════════════════════════

import json, re
from pathlib import Path

DATA_DIR = Path("data")
PROC_DIR = Path("proc")
PROC_DIR.mkdir(exist_ok=True)

# ── Check what fields are in your JSONL ────────────────────────────────────
sample_file = DATA_DIR / "en-train.jsonl"
with open(sample_file, encoding="utf-8") as f:
    sample = json.loads(f.readline())
print("Available JSONL fields:")
for k, v in sample.items():
    if k != "text":
        print(f"  {k}: {str(v)[:80]}")

def clean(t, n=900):
    return re.sub(r"\s+", " ", re.sub(r"[\x00-\x1f\x7f-\x9f]", " ", t)).strip()[:n]

def build_v11(text, pers, locs, date_str="", lang=""):
    p = clean(pers[0]) if pers else "UNKNOWN"
    l = clean(locs[0]) if locs else "UNKNOWN"
    date_tok = f"<DATE> {date_str} </DATE> " if date_str else ""
    lang_tok  = f"<LANG> {lang} </LANG> "    if lang    else ""
    return f"<P> {p} </P> <L> {l} </L> {date_tok}{lang_tok}{clean(text, 800)}"

AT_MAP   = {"FALSE":0, "PROBABLE":1, "TRUE":2}
ISAT_MAP = {"FALSE":0, "TRUE":1}

def load_jsonl_v11(path, lang):
    recs = []
    for line in open(path, encoding="utf-8"):
        doc = json.loads(line)
        # Extract date — try common field names
        date_str = ""
        for field in ["date", "pub_date", "year", "publication_date", "decade"]:
            if field in doc and doc[field]:
                date_str = str(doc[field])[:10]; break
        for pair in doc.get("sampled_pairs", []):
            at   = pair.get("at",   "FALSE")
            isat = pair.get("isAt", "FALSE")
            if at not in AT_MAP or isat not in ISAT_MAP: continue
            recs.append({
                "text":  build_v11(doc["text"], pair["pers_mentions_list"],
                                   pair["loc_mentions_list"], date_str, lang),
                "at":    AT_MAP[at],
                "isat":  ISAT_MAP[isat],
                "lang":  lang
            })
    return recs

print("\n── Building v11 dataset (with DATE + LANG tokens) ─────────────────")
for split, fmap in [
    ("train", {"en":"en-train", "fr":"fr-train", "de":"de-train"}),
    ("dev",   {"en":"en-dev",   "fr":"fr-dev",   "de":"de-dev"}),
]:
    all_r = []
    for lang, fname in fmap.items():
        r = load_jsonl_v11(DATA_DIR/f"{fname}.jsonl", lang)
        all_r.extend(r); print(f"  {lang} {split:5s}: {len(r):,}")
    with open(PROC_DIR/f"{split}_v11.jsonl", "w", encoding="utf-8") as f:
        for rec in all_r: f.write(json.dumps(rec, ensure_ascii=False)+"\n")
    print(f"✅ {split}: {len(all_r):,}\n")

# Show sample to verify DATE/LANG tokens appear
sample_rec = json.loads(open(PROC_DIR/"train_v11.jsonl").readline())
print("Sample input (first 200 chars):")
print(f"  {sample_rec['text'][:200]}")
print(f"\n✅ v11 dataset ready → proc/train_v11.jsonl")
print(f"   Now change PROC_DIR references in your training cell to use *_v11.jsonl")
print(f"   Also add <DATE> </DATE> <LANG> </LANG> to SPECIAL tokens list")


Available JSONL fields:
  document_id: sn83026170-1820-01-10-a-i0001
  media: {'publication_title': 'Alexandria Gazette & Daily Advertiser', 'time_period': '1
  source: v2.1/hipe2020/en/HIPE-2022-v2.1-hipe2020-dev-en.tsv
  language: en
  date: 1820-01-10
  sampled_pairs: [{'pers_entity_id': 'sn83026170-1820-01-10-a-i0001-NIL_mr_pain_wareing', 'pers_w

── Building v11 dataset (with DATE + LANG tokens) ─────────────────
  en train: 496
  fr train: 4,450
  de train: 1,224
✅ train: 6,170

  en dev  : 151
  fr dev  : 1,498
  de dev  : 432
✅ dev: 2,081

Sample input (first 200 chars):
  <P> Mr. Pain Wareing </P> <L> Rappahannock </L> <DATE> 1820-01-10 </DATE> <LANG> en </LANG> Was Committed, T O the /ail for the county of Alexandria, D. C. on the 23d inst. a negro man, who called him

✅ v11 dataset ready → proc/train_v11.jsonl
   Now change PROC_DIR references in your training cell to use *_v11.jsonl
   Also add <DATE> </DATE> <LANG> </LANG> to SPECIAL tokens list


In [7]:
# ═══════════════════════════════════════════════════════════════════
#  MHIPEX v11 — hmBERT only, enriched inputs (DATE + LANG tokens)
#  Dataset: proc/train_v11.jsonl & proc/dev_v11.jsonl
#  New special tokens: <DATE> </DATE> <LANG> </LANG> added
#  Expected MR: 0.55–0.60 | Runtime: ~42 min on T4 x2
# ═══════════════════════════════════════════════════════════════════

import os, math, json, re, random, copy, gc, time
os.environ["PYTORCH_ALLOC_CONF"]     = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from pathlib import Path
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from transformers import AutoModel, AutoTokenizer, get_cosine_schedule_with_warmup
from sklearn.metrics import recall_score, classification_report
from tqdm import tqdm

SEED = 42
def set_seed(s=SEED):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
set_seed()
torch.backends.cuda.matmul.allow_tf32 = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_GPU  = torch.cuda.device_count()
print(f"Device: {DEVICE} | GPUs: {N_GPU} | CUDA: {torch.version.cuda}")

AT_MAP     = {"FALSE":0,"PROBABLE":1,"TRUE":2}
ISAT_MAP   = {"FALSE":0,"TRUE":1}
AT_NAMES   = ["FALSE","PROBABLE","TRUE"]
ISAT_NAMES = ["FALSE","TRUE"]

# v11: 8 special tokens (4 original + 4 new for DATE/LANG)
SPECIAL = ["<P>","</P>","<L>","</L>","<DATE>","</DATE>","<LANG>","</LANG>"]

AT_W   = torch.tensor([0.80,1.50,2.40], dtype=torch.float32)
ISAT_W = torch.tensor([0.70,2.60],      dtype=torch.float32)
WD     = 0.01

PROC_DIR = Path("proc"); OUT_DIR = Path("out"); OUT_DIR.mkdir(exist_ok=True)

CFG = {
    "name":  "dbmdz/bert-base-historic-multilingual-cased",
    "bs":32, "accum":2, "lr":8e-6, "decay":0.90,
    "pat":8, "ep":35, "wu":0.12, "maxlen":256, "drop":0.15, "clip":0.5,
}

class HIPEDataset(Dataset):
    def __init__(self, path, tok, maxlen=256):
        self.data = [json.loads(l) for l in open(path, encoding="utf-8")]
        self.tok = tok; self.maxlen = maxlen
    def __len__(self): return len(self.data)
    def __getitem__(self, i):
        d = self.data[i]
        enc = self.tok(d["text"], max_length=self.maxlen, truncation=True,
                       padding="max_length", return_tensors="pt")
        return {"input_ids": enc["input_ids"].squeeze(0),
                "attention_mask": enc["attention_mask"].squeeze(0),
                "at_label":   torch.tensor(d["at"],   dtype=torch.long),
                "isat_label": torch.tensor(d["isat"], dtype=torch.long)}

def balanced_sampler(ds):
    labels = np.array([r["at"] for r in ds.data])
    counts = np.bincount(labels, minlength=3).astype(float)
    return WeightedRandomSampler([1.0/counts[l] for l in labels], len(labels), replacement=True)

class FocalLoss(nn.Module):
    def __init__(self, weight=None, gamma=2.0):
        super().__init__(); self.gamma = gamma
        self.register_buffer("weight", weight)
    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.weight, reduction="none")
        return (((1 - torch.exp(-ce)) ** self.gamma) * ce).mean()

class MHIPEXModel(nn.Module):
    def __init__(self, name, n_new_tok, dropout=0.15):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(name)
        self.encoder.resize_token_embeddings(self.encoder.config.vocab_size + n_new_tok)
        h = self.encoder.config.hidden_size
        self.layer_norm = nn.LayerNorm(h); self.dropout = nn.Dropout(dropout)
        self.head_at   = nn.Sequential(nn.Linear(h,h//2), nn.GELU(), nn.Dropout(dropout), nn.Linear(h//2,3))
        self.head_isat = nn.Sequential(nn.Linear(h,h//2), nn.GELU(), nn.Dropout(dropout), nn.Linear(h//2,2))
    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = self.layer_norm(self.dropout(out.last_hidden_state[:, 0, :]))
        return self.head_at(cls), self.head_isat(cls)

def layerwise_groups(model, base_lr, decay):
    raw = model.module if hasattr(model, "module") else model
    enc = raw.encoder; groups = []
    layers = getattr(getattr(enc, "encoder", enc), "layer", None)
    if layers:
        n = len(layers)
        for i, layer in enumerate(layers):
            groups.append({"params": list(layer.parameters()), "lr": base_lr*(decay**(n-1-i))})
    if hasattr(enc, "embeddings"):
        groups.append({"params": list(enc.embeddings.parameters()),
                       "lr": base_lr*(decay**(len(layers) if layers else 12))})
    for head in [raw.head_at, raw.head_isat, raw.layer_norm]:
        groups.append({"params": list(head.parameters()), "lr": base_lr})
    return groups

def macro_recall(at_t, at_p, is_t, is_p):
    r1 = recall_score(at_t, at_p, average="macro", zero_division=0)
    r2 = recall_score(is_t, is_p, average="macro", zero_division=0)
    return round((r1+r2)/2, 4), round(r1, 4), round(r2, 4)

# ════════════════════════════════════════════════
print(f"\n{'█'*60}")
print(f"  MHIPEX v11 | hmBERT + DATE/LANG enriched inputs")
print(f"  🎯 Target: beat 0.5382  | ⏱ ~42 min")
print(f"{'█'*60}")

gc.collect(); torch.cuda.empty_cache()
print(f"  GPU free: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")
set_seed(); t0 = time.time()

tok = AutoTokenizer.from_pretrained(CFG["name"])
n   = tok.add_special_tokens({"additional_special_tokens": SPECIAL})
print(f"  Added {n} special tokens")
for st in SPECIAL:
    ids = tok.encode(st, add_special_tokens=False)
    print(f"    '{st}' → {ids} ({'✅' if len(ids)==1 else '⚠ split'})")

# Use v11 dataset files
tr_ds = HIPEDataset(PROC_DIR/"train_v11.jsonl", tok, CFG["maxlen"])
dv_ds = HIPEDataset(PROC_DIR/"dev_v11.jsonl",   tok, CFG["maxlen"])
TL = DataLoader(tr_ds, batch_size=CFG["bs"], sampler=balanced_sampler(tr_ds), num_workers=2, pin_memory=True)
DL = DataLoader(dv_ds, batch_size=CFG["bs"], shuffle=False, num_workers=2, pin_memory=True)

model = MHIPEXModel(CFG["name"], n_new_tok=n, dropout=CFG["drop"]).to(DEVICE)
if N_GPU > 1: model = nn.DataParallel(model); print(f"  DataParallel:{N_GPU} GPUs ✅")

fl_at   = FocalLoss(weight=AT_W.to(DEVICE),   gamma=2.0)
fl_isat = FocalLoss(weight=ISAT_W.to(DEVICE), gamma=2.0)

groups = layerwise_groups(model, CFG["lr"], CFG["decay"])
opt    = torch.optim.AdamW(groups, weight_decay=WD)
print(f"  Layer-wise LR: {len(groups)} groups ✅")

steps  = math.ceil(len(TL)/CFG["accum"]) * CFG["ep"]
warmup = int(steps * CFG["wu"])
sch    = get_cosine_schedule_with_warmup(opt, warmup, steps)
scaler = torch.amp.GradScaler("cuda")
opt.zero_grad(set_to_none=True)
print(f"  Steps/ep:{math.ceil(len(TL)/CFG['accum'])} | Total:{steps} | Warmup:{warmup}\n")

best_mr=0.0; no_imp=0; best_state=None

for ep in range(1, CFG["ep"]+1):
    model.train(); ep_loss=0.0; nb=0
    for bi, batch in enumerate(tqdm(TL, desc=f"  Ep{ep:02d} TRAIN", leave=False)):
        ids  = batch["input_ids"].to(DEVICE); mask = batch["attention_mask"].to(DEVICE)
        al   = batch["at_label"].to(DEVICE);  il   = batch["isat_label"].to(DEVICE)
        with torch.amp.autocast("cuda"):
            ao, io = model(ids, mask)
            loss   = (fl_at(ao, al) + fl_isat(io, il)) / CFG["accum"]
        scaler.scale(loss).backward()
        if (bi+1) % CFG["accum"] == 0 or (bi+1) == len(TL):
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CFG["clip"])
            scaler.step(opt); scaler.update(); sch.step(); opt.zero_grad(set_to_none=True)
        if not torch.isnan(loss): ep_loss += loss.item()*CFG["accum"]; nb += 1

    avg = round(ep_loss/max(nb,1), 4)
    model.eval(); at_t,at_p,is_t,is_p=[],[],[],[]
    with torch.no_grad():
        for batch in tqdm(DL, desc=f"  Ep{ep:02d} EVAL ", leave=False):
            with torch.amp.autocast("cuda"):
                ao, io = model(batch["input_ids"].to(DEVICE), batch["attention_mask"].to(DEVICE))
            at_p.extend(ao.argmax(1).cpu().tolist()); at_t.extend(batch["at_label"].tolist())
            is_p.extend(io.argmax(1).cpu().tolist()); is_t.extend(batch["isat_label"].tolist())

    mr, mr_at, mr_is = macro_recall(at_t, at_p, is_t, is_p)
    print(f"\n  {'─'*58}")
    print(f"  Ep{ep:02d}|Loss:{avg:.4f}|MR:{mr:.4f}(at={mr_at:.4f},isAt={mr_is:.4f})|{round((time.time()-t0)/60,1)}min")
    print(f"  {'─'*58}")

    if mr > best_mr:
        best_mr=mr; no_imp=0
        raw = model.module if hasattr(model,"module") else model
        best_state = copy.deepcopy(raw.state_dict())
        print(f"  ✅ Best MR={mr:.4f} {'🔥 BEATS v8!' if mr > 0.5382 else ''}")
    else:
        no_imp += 1; print(f"  ↘ No improvement ({no_imp}/{CFG['pat']})")
        if no_imp >= CFG["pat"]: print(f"  ⏹ Early stop ep{ep}"); break

    if ep % 5 == 0:
        print(classification_report(at_t, at_p, target_names=AT_NAMES, zero_division=0))
        print(classification_report(is_t, is_p, target_names=ISAT_NAMES, zero_division=0))

# Save
sd = OUT_DIR/"hmbert_v11"; sd.mkdir(exist_ok=True)
if best_state:
    raw = model.module if hasattr(model,"module") else model
    raw.load_state_dict(best_state)
    torch.save(best_state, sd/"best_model.pt"); tok.save_pretrained(sd)

total = round((time.time()-t0)/60, 1)
print(f"\n  💾 v11 hmBERT | Best MR:{best_mr:.4f} | {total}min")

# Apply Fix 1 + Fix 2 automatically on top of v11 predictions
del model, opt, sch, scaler, TL, DL; gc.collect(); torch.cuda.empty_cache()

print(f"\n{'='*60}")
print(f"  v11 FINAL SCORECARD")
print(f"{'='*60}")
print(f"  v8 best (previous):  0.5382")
print(f"  v11 hmBERT:          {best_mr:.4f}  {'🔥 NEW BEST!' if best_mr > 0.5382 else '← check threshold calibration next'}")
print(f"  Gap to SOTA (0.69):  {0.6900 - best_mr:.4f}")
print(f"{'='*60}")

print(f"\n✅ Model saved → out/hmbert_v11/")
print(f"   Run Fix 2 (threshold calibration) on this model next for additional gain")

Device: cuda | GPUs: 2 | CUDA: 12.8

████████████████████████████████████████████████████████████
  MHIPEX v11 | hmBERT + DATE/LANG enriched inputs
  🎯 Target: beat 0.5382  | ⏱ ~42 min
████████████████████████████████████████████████████████████
  GPU free: 12.52 GB


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


  Added 8 special tokens
    '<P>' → [32000] (✅)
    '</P>' → [32001] (✅)
    '<L>' → [32002] (✅)
    '</L>' → [32003] (✅)
    '<DATE>' → [32004] (✅)
    '</DATE>' → [32005] (✅)
    '<LANG>' → [32006] (✅)
    '</LANG>' → [32007] (✅)
  DataParallel:2 GPUs ✅
  Layer-wise LR: 16 groups ✅
  Steps/ep:97 | Total:3395 | Warmup:407




  ──────────────────────────────────────────────────────────
  Ep01|Loss:1.4020|MR:0.4167(at=0.3333,isAt=0.5000)|1.2min
  ──────────────────────────────────────────────────────────
  ✅ Best MR=0.4167 



  ──────────────────────────────────────────────────────────
  Ep02|Loss:1.2844|MR:0.4167(at=0.3333,isAt=0.5000)|2.4min
  ──────────────────────────────────────────────────────────
  ↘ No improvement (1/8)


...


  ──────────────────────────────────────────────────────────
  Ep17|Loss:0.5892|MR:0.5167(at=0.4034,isAt=0.6301)|20.4min
  ──────────────────────────────────────────────────────────
  ↘ No improvement (4/8)



  ──────────────────────────────────────────────────────────
  Ep18|Loss:0.5646|MR:0.5220(at=0.4102,isAt=0.6338)|21.6min
  ──────────────────────────────────────────────────────────
  ↘ No improvement (5/8)



  ──────────────────────────────────────────────────────────
  Ep19|Loss:0.5284|MR:0.5202(at=0.4181,isAt=0.6224)|22.8min
  ──────────────────────────────────────────────────────────
  ↘ No improvement (6/8)



  ──────────────────────────────────────────────────────────
  Ep20|Loss:0.5317|MR:0.5092(at=0.3957,isAt=0.6227)|23.9min
  ──────────────────────────────────────────────────────────
  ↘ No improvement (7/8)
              precision    recall  f1-score   support

       FALSE       0.67      0.10      0.17      1264
    PROBABLE       0.28      0.61      0.38       568
        TRUE       0.18      0.48      0.26       249

    accuracy                           0.28      2081
   macro avg       0.38      0.40      0.27      2081
weighted avg       0.50      0.28      0.24      2081

              precision    recall  f1-score   support

       FALSE       0.95      0.67      0.78      1907
        TRUE       0.14      0.57      0.22       174

    accuracy                           0.66      2081
   macro avg       0.54      0.62      0.50      2081
weighted avg       0.88      0.66      0.74      2081




  ──────────────────────────────────────────────────────────
  Ep21|Loss:0.4974|MR:0.5114(at=0.4029,isAt=0.6198)|25.1min
  ──────────────────────────────────────────────────────────
  ↘ No improvement (8/8)
  ⏹ Early stop ep21

  💾 v11 hmBERT | Best MR:0.5250 | 25.2min

  v11 FINAL SCORECARD
  v8 best (previous):  0.5382
  v11 hmBERT:          0.5250  ← check threshold calibration next
  Gap to SOTA (0.69):  0.1650

✅ Model saved → out/hmbert_v11/
   Run Fix 2 (threshold calibration) on this model next for additional gain


In [8]:
# ══════════════════════════════════════════════════════════════════
#  CALIBRATE v11 hmBERT (DATE/LANG model)
#  Uses saved model from: out/hmbert_v11/
#  Goal: see if threshold calibration can beat 0.5382
# ══════════════════════════════════════════════════════════════════

import json, gc
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer
from sklearn.metrics import recall_score
from itertools import product
from tqdm import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AT_NAMES   = ["FALSE", "PROBABLE", "TRUE"]
ISAT_NAMES = ["FALSE", "TRUE"]
SPECIAL    = ["<P>", "</P>", "<L>", "</L>", "<DATE>", "</DATE>", "<LANG>", "</LANG>"]
PROC_DIR   = Path("proc")
OUT_DIR    = Path("out")
MODEL_DIR  = OUT_DIR / "hmbert_v11"
MODEL_NAME = "dbmdz/bert-base-historic-multilingual-cased"

class HIPEDataset(Dataset):
    def __init__(self, path, tok, maxlen=256):
        self.data = [json.loads(l) for l in open(path, encoding="utf-8")]
        self.tok = tok; self.maxlen = maxlen
    def __len__(self): return len(self.data)
    def __getitem__(self, i):
        d = self.data[i]
        enc = self.tok(d["text"], max_length=self.maxlen, truncation=True,
                       padding="max_length", return_tensors="pt")
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "at_label": torch.tensor(d["at"], dtype=torch.long),
            "isat_label": torch.tensor(d["isat"], dtype=torch.long),
            "lang": d.get("lang", "xx")
        }

class MHIPEXModel(nn.Module):
    def __init__(self, name, n_new_tok, dropout=0.15):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(name)
        self.encoder.resize_token_embeddings(self.encoder.config.vocab_size + n_new_tok)
        h = self.encoder.config.hidden_size
        self.layer_norm = nn.LayerNorm(h)
        self.dropout = nn.Dropout(dropout)
        self.head_at   = nn.Sequential(nn.Linear(h,h//2), nn.GELU(), nn.Dropout(dropout), nn.Linear(h//2,3))
        self.head_isat = nn.Sequential(nn.Linear(h,h//2), nn.GELU(), nn.Dropout(dropout), nn.Linear(h//2,2))
    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = self.layer_norm(self.dropout(out.last_hidden_state[:, 0, :]))
        return self.head_at(cls), self.head_isat(cls)

def macro_recall(at_t, at_p, is_t, is_p):
    r1 = recall_score(at_t, at_p, average="macro", zero_division=0)
    r2 = recall_score(is_t, is_p, average="macro", zero_division=0)
    return round((r1+r2)/2, 4), round(r1,4), round(r2,4)

print("="*64)
print("  CALIBRATING v11 hmBERT")
print("="*64)
print(f"  Model dir: {MODEL_DIR}")
print(f"  Device: {DEVICE}")

# Load tokenizer and dev set
print("\n  Loading tokenizer and dev_v11...")
tok = AutoTokenizer.from_pretrained(str(MODEL_DIR))
tok.add_special_tokens({"additional_special_tokens": SPECIAL})

ds = HIPEDataset(PROC_DIR / "dev_v11.jsonl", tok, 256)
dl = DataLoader(ds, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

# Load model
print("  Loading saved v11 model...")
model = MHIPEXModel(MODEL_NAME, n_new_tok=len(SPECIAL)).to(DEVICE)
state = torch.load(str(MODEL_DIR / "best_model.pt"), map_location=DEVICE)
model.load_state_dict(state)
model.eval()

# Extract probabilities
print("  Extracting probabilities...")
at_probs_all, is_probs_all = [], []
at_true, is_true, langs = [], [], []
with torch.no_grad():
    for b in tqdm(dl, desc="  Inference"):
        with torch.amp.autocast("cuda"):
            ao, io = model(b["input_ids"].to(DEVICE), b["attention_mask"].to(DEVICE))
        at_probs_all.append(F.softmax(ao, dim=1).cpu().numpy())
        is_probs_all.append(F.softmax(io, dim=1).cpu().numpy())
        at_true.extend(b["at_label"].tolist())
        is_true.extend(b["isat_label"].tolist())
        langs.extend(b["lang"])

at_probs = np.concatenate(at_probs_all)
is_probs = np.concatenate(is_probs_all)
del model; gc.collect(); torch.cuda.empty_cache()

# Baseline argmax
at_pred_base = at_probs.argmax(axis=1)
is_pred_base = is_probs.argmax(axis=1)
mr_base, mr_at_base, mr_is_base = macro_recall(at_true, at_pred_base, is_true, is_pred_base)
print(f"\n  Baseline argmax MR: {mr_base:.4f} (at={mr_at_base:.4f}, isAt={mr_is_base:.4f})")

# Search thresholds for AT
print("\n  Grid searching AT thresholds...")
thresholds = [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]
best_at_score = -1
best_t_at = (None, None)
best_at_pred = None

for t_prob, t_true in product(thresholds, thresholds):
    preds = []
    for p in at_probs:
        if p[2] >= t_true:
            preds.append(2)
        elif p[1] >= t_prob:
            preds.append(1)
        else:
            preds.append(0)
    score = recall_score(at_true, preds, average="macro", zero_division=0)
    if score > best_at_score:
        best_at_score = score
        best_t_at = (t_prob, t_true)
        best_at_pred = preds

print(f"  Best AT thresholds: PROBABLE>={best_t_at[0]:.2f}, TRUE>={best_t_at[1]:.2f}")
print(f"  Best AT macro recall: {best_at_score:.4f}")

# Search thresholds for isAt with logical constraint
print("\n  Grid searching isAt thresholds + logical constraint...")
best_is_score = -1
best_t_is = None
best_is_pred = None

for t_is in [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65]:
    preds = []
    for i, p in enumerate(is_probs):
        if best_at_pred[i] == 0:   # logical constraint
            preds.append(0)
        else:
            preds.append(int(p[1] >= t_is))
    score = recall_score(is_true, preds, average="macro", zero_division=0)
    if score > best_is_score:
        best_is_score = score
        best_t_is = t_is
        best_is_pred = preds

print(f"  Best isAt threshold: TRUE>={best_t_is:.2f}")
print(f"  Best isAt macro recall: {best_is_score:.4f}")

# Final calibrated score
mr_final, mr_at_final, mr_is_final = macro_recall(at_true, best_at_pred, is_true, best_is_pred)
print(f"\n{'='*64}")
print(f"  v11 Baseline argmax:      MR = {mr_base:.4f}")
print(f"  v11 Calibrated final:     MR = {mr_final:.4f}")
print(f"  Gain over v11 baseline:  +{mr_final - mr_base:.4f}")
print(f"  Best old score (v8):      MR = 0.5382")
print(f"  Status: {'🔥 NEW BEST!' if mr_final > 0.5382 else 'still below v8'}")
print(f"{'='*64}")

# Per-language view
print("\n  PER-LANGUAGE (calibrated)")
for lg in sorted(set(langs)):
    idx = [i for i, x in enumerate(langs) if x == lg]
    lg_mr, lg_at, lg_is = macro_recall(
        [at_true[i] for i in idx], [best_at_pred[i] for i in idx],
        [is_true[i] for i in idx], [best_is_pred[i] for i in idx]
    )
    print(f"  {lg}: MR={lg_mr:.4f} | at={lg_at:.4f} | isAt={lg_is:.4f} | N={len(idx)}")

# Save calibrated predictions
recs = [json.loads(l) for l in open(PROC_DIR / "dev_v11.jsonl", encoding="utf-8")]
out_path = OUT_DIR / "predictions_v11_calibrated.jsonl"
with open(out_path, "w", encoding="utf-8") as f:
    for i, rec in enumerate(recs):
        f.write(json.dumps({
            "lang": rec.get("lang", "xx"),
            "at_true": AT_NAMES[at_true[i]],
            "at_pred": AT_NAMES[best_at_pred[i]],
            "isat_true": ISAT_NAMES[is_true[i]],
            "isat_pred": ISAT_NAMES[best_is_pred[i]],
            "at_correct": best_at_pred[i] == at_true[i],
            "isat_correct": best_is_pred[i] == is_true[i],
            "at_probs": [round(float(x), 6) for x in at_probs[i]],
            "isat_probs": [round(float(x), 6) for x in is_probs[i]],
        }, ensure_ascii=False) + "\n")

print(f"\n✅ Saved calibrated predictions → {out_path}")
print("✅ Run complete")

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


  CALIBRATING v11 hmBERT
  Model dir: out/hmbert_v11
  Device: cuda

  Loading tokenizer and dev_v11...
  Loading saved v11 model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


  Extracting probabilities...


  Inference: 100%|██████████| 66/66 [00:09<00:00,  6.75it/s]



  Baseline argmax MR: 0.5236 (at=0.4090, isAt=0.6382)

  Grid searching AT thresholds...
  Best AT thresholds: PROBABLE>=0.50, TRUE>=0.30
  Best AT macro recall: 0.4308

  Grid searching isAt thresholds + logical constraint...
  Best isAt threshold: TRUE>=0.55
  Best isAt macro recall: 0.6422

  v11 Baseline argmax:      MR = 0.5236
  v11 Calibrated final:     MR = 0.5365
  Gain over v11 baseline:  +0.0129
  Best old score (v8):      MR = 0.5382
  Status: still below v8

  PER-LANGUAGE (calibrated)
  de: MR=0.5395 | at=0.4436 | isAt=0.6354 | N=432
  en: MR=0.5840 | at=0.4325 | isAt=0.7354 | N=151
  fr: MR=0.5250 | at=0.4226 | isAt=0.6275 | N=1498

✅ Saved calibrated predictions → out/predictions_v11_calibrated.jsonl
✅ Run complete
